# Femoral Shaft Fracture Detection and AO Classification System 🦴🐈🐕

## Overview

This notebook implements a **Faster R-CNN-based deep learning system** for automated detection and AO-based severity classification of femoral shaft fractures in canine and feline radiographs.

### Objectives

- Detect femoral shaft fractures with bounding box localization
- Classify fractures according to the AO/OTA classification system
- Provide reliable diagnostic support for veterinary practitioners

### Architecture

- **Model**: Faster R-CNN with ResNet-50 + FPN backbone
- **Transfer Learning**: Pre-trained on COCO dataset
- **Multi-task**: Joint detection and classification

### AO Classification Categories

| Class | Label | Description |
|-------|-------|-------------|
| 0 | No Fracture | Intact femoral shaft |
| 1 | 32-A | Simple fracture (two fragments) |
| 2 | 32-B | Wedge fracture (butterfly fragment) |
| 3 | 32-C | Complex/comminuted fracture (multiple fragments) |

### Key Features

- **End-to-End Fracture Detection and AO-Based Classification**: A Faster R-CNN framework that simultaneously localizes femoral shaft fractures using bounding boxes and classifies fracture severity into AO categories (No Fracture, 32-A, 32-B, 32-C).
- **AO-Standard Multi-Class Severity Labeling**: Implements clinically standardized AO fracture classes, enabling consistent and interpretable severity assessment across canine and feline radiographs.
- **Selective CLAHE Contrast Enhancement**: Enhances radiograph contrast using CLAHE with `clip_limit=3.0` and `tile_grid_size=(8,8)`, applied unconditionally (`p=1.0`) to all images during training, validation, and testing.
- **Augmentation with Bounding Box Integrity**: Uses Albumentations for geometric and photometric augmentation with automatic bounding box transformation, ensuring precise annotation alignment after preprocessing.
- **Comprehensive Detection and Classification Evaluation**: Assesses performance using IoU and mAP@0.5 for fracture localization, and Accuracy, Precision, Sensitivity, Specificity, and F1-score for AO-based classification.

### Requirements

- **Runtime**: GPU (NVIDIA T4, A100, or equivalent)
- **Python**: 3.10+
- **PyTorch**: 2.0+
- **Storage**: Google Drive (for dataset and checkpoints)

### Expected Directory Structure

```
Google Drive/MyDrive/Thesis/Thesis 2/System Files/system/
├── images/
│   ├── train/
│   ├── val/
│   └── test/
├── annotations/
│   ├── instances_train.json
│   ├── instances_val.json
│   └── instances_test.json
├── models/        # Model checkpoints saved here
└── logs/          # TensorBoard logs saved here
```

---


## 1️⃣ Environment Setup

### Purpose
Install all required Python packages for:
- Image preprocessing and augmentation
- Deep learning model training
- Evaluation metrics computation
- Visualization and monitoring

### Packages Installed

| Package | Version | Purpose |
|---------|---------|---------|
| `albumentations` | 1.3.1 | Data augmentation with bounding box transformation |
| `pycocotools` | latest | COCO format dataset handling and mAP calculation |
| `opencv-python-headless` | latest | Image preprocessing (CLAHE, resizing, normalization) |
| `pandas` | latest | Data manipulation and metrics logging |
| `seaborn` | latest | Statistical visualization (confusion matrix) |
| `matplotlib` | latest | Plotting training curves and sample predictions |
| `scikit-learn` | latest | Classification metrics (precision, recall, F1) |
| `tqdm` | latest | Progress bars for training loops |
| `tensorboard` | latest | Real-time training monitoring |

### Installation Time
⏱ Approximately 1–2 minutes

---


In [ ]:
# Install required packages quietly
!pip install -q albumentations==1.3.1 pycocotools opencv-python-headless pandas seaborn matplotlib scikit-learn tqdm tensorboard

print("All dependencies installed successfully!")


## 2️⃣ Mount Google Drive

### Purpose
Connect to Google Drive to access:
- Training, validation, and test datasets
- Saved model checkpoints
- TensorBoard logs

### What Happens
1. Google Colab will prompt you to authenticate
2. Click the authentication link
3. Select your Google account
4. Grant access when prompted

### Mount Point
Your Google Drive will be accessible at: `/content/drive/MyDrive/`

---


In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

print("Google Drive mounted successfully!")
print("Access your files at: /content/drive/MyDrive/")


## 3️⃣ Dataset Distribution

### Purpose
Inspect the dataset before training to verify:
- Total image counts per split (train / val / test)
- Species distribution (dogs vs. cats)
- AO class distribution per species and split
- Background-only images (no bounding box annotations)

### What This Cell Does
Reads the COCO-format annotation JSON files for each split and prints:
- Per-split image counts and species breakdown
- AO class frequency per species
- A detailed table showing Background, 32-A, 32-B, and 32-C counts
- Train / val / test percentage split

> **Note:** AO classes are counted per image, not per bounding box. An image may contain multiple fracture types, so class counts may not sum to the total image count.

---


In [ ]:
#!/usr/bin/env python3
import os
import json
from collections import defaultdict

# ─── Paths ──────────────────────────────────────────────────────────────────
BASE_PATH = "/content/drive/MyDrive/Thesis/Thesis 2/System Files/system"
SPLITS = ["train", "val", "test"]

# AO class mapping (must match COCO category IDs)
AO_CLASSES = {
    1: "32-A (Simple)",
    2: "32-B (Wedge)",
    3: "32-C (Complex)",
}


def count_images_species_and_classes():
    total_images = 0
    species_totals = defaultdict(int)
    background_only = defaultdict(int)
    species_class_counts = defaultdict(lambda: defaultdict(int))

    split_totals = defaultdict(int)
    split_species_totals = defaultdict(lambda: defaultdict(int))

    print("=" * 90)
    print("DATASET IMAGE, SPECIES, AND AO CLASS DISTRIBUTION")
    print("=" * 90)
    print()

    for split in SPLITS:
        ann_file = os.path.join(BASE_PATH, "annotations", f"instances_{split}.json")

        if not os.path.exists(ann_file):
            print(f"WARNING: {ann_file} not found — skipping {split} set")
            print()
            continue

        with open(ann_file, "r") as f:
            data = json.load(f)

        image_id_to_filename = {img["id"]: img["file_name"] for img in data.get("images", [])}

        # Map image_id → set of AO category IDs
        image_classes = defaultdict(set)
        for ann in data.get("annotations", []):
            cid = ann["category_id"]
            if cid in AO_CLASSES:
                image_classes[ann["image_id"]].add(cid)

        split_species = defaultdict(int)
        split_background = defaultdict(int)
        split_species_classes = defaultdict(lambda: defaultdict(int))

        for image_id, filename in image_id_to_filename.items():
            if "dog" in filename.lower():
                species = "Dogs"
            elif "cat" in filename.lower():
                species = "Cats"
            else:
                continue

            split_species[species] += 1
            split_species_totals[split][species] += 1
            species_totals[species] += 1
            split_totals[split] += 1
            total_images += 1

            if image_id not in image_classes:
                split_background[species] += 1
                background_only[species] += 1
            else:
                for cid in image_classes[image_id]:
                    cls_name = AO_CLASSES[cid]
                    split_species_classes[species][cls_name] += 1
                    species_class_counts[species][cls_name] += 1

        # Per-split summary
        split_total = sum(split_species.values())
        print(f"{split.upper()} SET:")
        print(f"  Total images: {split_total}")
        for species in ["Dogs", "Cats"]:
            count = split_species[species]
            pct = count / split_total * 100 if split_total else 0
            print(f"  {species}: {count} ({pct:.1f}%)")
            print(f"    Background-only: {split_background[species]}")
            for cls in AO_CLASSES.values():
                print(f"    {cls}: {split_species_classes[species].get(cls, 0)}")
        print()

    # Overall summary
    print("=" * 90)
    print("OVERALL SUMMARY")
    print("=" * 90)
    print(f"Total images: {total_images}")
    print()
    for species in ["Dogs", "Cats"]:
        pct = species_totals[species] / total_images * 100 if total_images else 0
        print(f"{species}: {species_totals[species]} ({pct:.1f}%)")
        print(f"  Background-only: {background_only[species]}")
        for cls in AO_CLASSES.values():
            print(f"  {cls}: {species_class_counts[species].get(cls, 0)}")
        print()

    # Train / val / test split percentages
    print("=" * 90)
    print("TRAIN / VAL / TEST SPLIT DISTRIBUTION")
    print("=" * 90)
    for split in SPLITS:
        count = split_totals[split]
        pct = count / total_images * 100 if total_images else 0
        print(f"{split.upper():5s} set: {count:4d} images ({pct:5.1f}%)")
        for species in ["Dogs", "Cats"]:
            sc = split_species_totals[split][species]
            sp = sc / count * 100 if count else 0
            print(f"  {species}: {sc:4d} ({sp:5.1f}% of {split})")
    print()
    print(f"{'TOTAL':5s}     {total_images:4d} images (100.0%)")
    print()

    # Detailed breakdown table
    print("=" * 90)
    print("DETAILED BREAKDOWN TABLE")
    print("=" * 90)
    print()
    print(f"{'Split':<8} {'Species':<8} {'Total':<8} {'BG-Only':<10} {'32-A':<8} {'32-B':<8} {'32-C':<8}")
    print("-" * 90)

    for split in SPLITS:
        ann_file = os.path.join(BASE_PATH, "annotations", f"instances_{split}.json")
        if not os.path.exists(ann_file):
            continue

        with open(ann_file, "r") as f:
            data = json.load(f)

        image_id_to_filename = {img["id"]: img["file_name"] for img in data.get("images", [])}
        image_classes = defaultdict(set)
        for ann in data.get("annotations", []):
            cid = ann["category_id"]
            if cid in AO_CLASSES:
                image_classes[ann["image_id"]].add(cid)

        for species in ["Dogs", "Cats"]:
            total = split_species_totals[split][species]
            bg = class_a = class_b = class_c = 0
            for image_id, filename in image_id_to_filename.items():
                is_species = (
                    ("dog" in filename.lower() and species == "Dogs") or
                    ("cat" in filename.lower() and species == "Cats")
                )
                if not is_species:
                    continue
                if image_id not in image_classes:
                    bg += 1
                else:
                    if 1 in image_classes[image_id]: class_a += 1
                    if 2 in image_classes[image_id]: class_b += 1
                    if 3 in image_classes[image_id]: class_c += 1
            print(f"{split:<8} {species:<8} {total:<8} {bg:<10} {class_a:<8} {class_b:<8} {class_c:<8}")

    print("-" * 90)
    for species in ["Dogs", "Cats"]:
        total = species_totals[species]
        bg = background_only[species]
        a = species_class_counts[species].get("32-A (Simple)", 0)
        b = species_class_counts[species].get("32-B (Wedge)", 0)
        c = species_class_counts[species].get("32-C (Complex)", 0)
        print(f"{'TOTAL':<8} {species:<8} {total:<8} {bg:<10} {a:<8} {b:<8} {c:<8}")

    print()
    print("=" * 90)
    print("NOTES")
    print("=" * 90)
    print("1. AO classes are counted per image, not per bounding box.")
    print("2. Background-only images have no bounding box annotations and are")
    print("   treated implicitly as background by the Faster R-CNN framework.")
    print("3. An image may contain multiple fracture types, so class counts")
    print("   may not sum to the total image count.")
    print("=" * 90)


if __name__ == "__main__":
    count_images_species_and_classes()


## 4️⃣ Complete Training Pipeline

### Purpose
This cell contains the **entire training pipeline** for the Faster R-CNN fracture detection system.

### Pipeline Components

#### 4.1 Configuration (`Config` class)
- Dataset paths for train / val / test splits
- Model hyperparameters: `lr=0.007`, `batch_size=16`, `epochs=50`, `momentum=0.9`, `weight_decay=0.004`
- AO class definitions: `{1: "32-A (Simple)", 2: "32-B (Wedge)", 3: "32-C (Complex)"}`
- Class-specific confidence thresholds: `{1: 0.65, 2: 0.45, 3: 0.65}`
- Directory paths for saving models and TensorBoard logs

#### 4.2 Data Preprocessing
- **Image loading**: OpenCV `imread` with BGR → RGB conversion (3 channels retained)
- **Bounding box format**: COCO `[x, y, w, h]` → Pascal VOC `[x1, y1, x2, y2]`, clipped to image boundaries
- **Resize strategy**: `LongestMaxSize(1024)` preserves aspect ratio
- **Padding**: `PadIfNeeded(1024×1024)` with black borders (`cv2.BORDER_CONSTANT`, `value=0`)
- **CLAHE enhancement**: Applied unconditionally (`p=1.0`) with `clip_limit=3.0`, `tile_grid_size=(8,8)`
- **Normalization**: `mean=(0.5,)`, `std=(0.5,)` → approximately `[-1, 1]` range

#### 4.3 Data Augmentation (Training Only)
- `HorizontalFlip` (p=0.5)
- `ShiftScaleRotate` (p=0.5): shift ±6.25%, scale ±10%, rotate ±20°
- `RandomBrightnessContrast` (p=0.3)
- Bounding boxes are automatically transformed via Albumentations `bbox_params`

#### 4.4 Dataset (`FemurDataset`)
- Custom PyTorch `Dataset` for COCO-format annotations
- Returns empty tensors for images with no valid annotations after preprocessing
- Wraps transforms in `try-except` to handle edge cases gracefully

#### 4.5 Model Architecture
- **Backbone**: Faster R-CNN ResNet-50 FPN V2 with COCO pre-trained weights
- **Detection head**: `FastRCNNPredictor` modified for 4-class output (background + 3 AO fracture types)
- **Total loss**: `sum(loss_dict.values())` across RPN + ROI losses

#### 4.6 Training Loop
- **Optimizer**: SGD with `lr=0.007`, `momentum=0.9`, `weight_decay=0.004`
- **LR Scheduler**: `CosineAnnealingLR` with `T_max=50`
- **Per-batch logging**: Batch loss logged to TensorBoard at `global_step = epoch × len(train_loader) + batch_idx`

#### 4.7 Validation Metrics (Per Epoch)

**Classification Metrics** (first detection per image used as class label):
- **Accuracy**: `accuracy_score(y_true, y_pred)`
- **Precision, Sensitivity (Recall), F1**: macro-averaged via `precision_recall_fscore_support`
- **Specificity**: Macro-averaged per-class TN / (TN + FP) using the 4×4 confusion matrix

**Detection Metrics**:
- Predictions filtered by class-specific confidence thresholds
- IoU computed with `torchvision.ops.box_iou`
- **mAP@0.5 (simplified)**: `matched_detections(IoU ≥ 0.5) / total_ground_truths`
- **Average IoU**: mean of all IoU scores ≥ 0.5

#### 4.8 Logging and Checkpointing
- TensorBoard scalars logged per epoch: Loss, LR, Accuracy, Precision, Sensitivity, Specificity, F1, mAP@0.5, Avg IoU
- Best model checkpoint saved when `current_f1 > best_f1`; includes model state, optimizer state, epoch, and all validation metrics

### Execution Steps

1. Verify dataset paths in the `Config` class match your Google Drive structure
2. Adjust hyperparameters if needed (learning rate, batch size, epochs)
3. Run the cell to start training
4. Monitor progress via console output and TensorBoard (Section 5)

### Training Time Estimate
- **GPU (T4)**: ~30–90 minutes
- **GPU (A100)**: ~15–45 minutes

---


In [ ]:
# Training Pipeline
"""
Femoral Shaft Fracture Detection System — Training Pipeline
===========================================================
"""

import os
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_iou
from pycocotools.coco import COCO
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score,
)
from tqdm.auto import tqdm
from collections import defaultdict
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    """Centralized configuration for the training pipeline."""

    # Dataset paths
    ROOT_DIR       = "/content/drive/MyDrive/Thesis/Thesis 2/System Files/system"
    TRAIN_IMG_DIR  = os.path.join(ROOT_DIR, "images/train")
    VAL_IMG_DIR    = os.path.join(ROOT_DIR, "images/val")
    TEST_IMG_DIR   = os.path.join(ROOT_DIR, "images/test")
    TRAIN_JSON     = os.path.join(ROOT_DIR, "annotations/instances_train.json")
    VAL_JSON       = os.path.join(ROOT_DIR, "annotations/instances_val.json")
    TEST_JSON      = os.path.join(ROOT_DIR, "annotations/instances_test.json")

    # Output directories
    MODEL_DIR            = os.path.join(ROOT_DIR, "models")
    MODEL_SAVE_PATH      = os.path.join(MODEL_DIR, "best_model.pth")
    RUN_DIR              = os.path.join(ROOT_DIR, "logs")
    TENSORBOARD_LOG_DIR  = os.path.join(
        RUN_DIR, f"fracture_detection_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    )
    VIZ_DIR = os.path.join(RUN_DIR, "visualizations")

    # Hardware
    DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_WORKERS = 0       # 0 avoids DataLoader worker crashes in Colab
    BATCH_SIZE  = 16
    IMAGE_SIZE  = 1024

    # Model hyperparameters
    NUM_CLASSES   = 4     # Background + 3 AO fracture types
    LEARNING_RATE = 0.007
    MOMENTUM      = 0.9
    WEIGHT_DECAY  = 0.004
    EPOCHS        = 50

    # Class-specific confidence thresholds
    CONF_THRESH = {
        1: 0.65,  # 32-A Simple  (strict)
        2: 0.45,  # 32-B Wedge   (lenient)
        3: 0.65,  # 32-C Complex (strict)
    }
    IOU_THRESHOLD = 0.5

    # Class labels
    AO_LABELS = {
        0: "No Fracture",
        1: "32-A (Simple)",
        2: "32-B (Wedge)",
        3: "32-C (Complex)",
    }


# Create output directories
os.makedirs(Config.MODEL_DIR, exist_ok=True)
os.makedirs(Config.RUN_DIR,   exist_ok=True)
os.makedirs(Config.VIZ_DIR,   exist_ok=True)

# Print system configuration
print("=" * 70)
print("SYSTEM CONFIGURATION")
print("=" * 70)
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("Warning: Running on CPU — training will be significantly slower")

print(f"\nConfiguration:")
print(f"  Batch Size:       {Config.BATCH_SIZE}")
print(f"  Image Resolution: {Config.IMAGE_SIZE}px")
print(f"  Maximum Epochs:   {Config.EPOCHS}")
print(f"  Learning Rate:    {Config.LEARNING_RATE}")
print(f"\nOutput Paths:")
print(f"  Models:     {Config.MODEL_DIR}")
print(f"  Logs:       {Config.RUN_DIR}")
print(f"  TensorBoard:{Config.TENSORBOARD_LOG_DIR}")
print("=" * 70 + "\n")


# ============================================================================
# DATA TRANSFORMS
# ============================================================================
def get_transforms(train=True):
    """
    Build the augmentation / preprocessing pipeline.

    Args:
        train (bool): If True, include training augmentations.

    Returns:
        albumentations.Compose: Transformation pipeline with bbox support.
    """
    transforms_list = [
        A.LongestMaxSize(max_size=Config.IMAGE_SIZE),
        A.PadIfNeeded(
            min_height=Config.IMAGE_SIZE,
            min_width=Config.IMAGE_SIZE,
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
        ),
        A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=1.0),
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2(),
    ]

    if train:
        transforms_list.insert(2, A.HorizontalFlip(p=0.5))
        transforms_list.insert(3, A.ShiftScaleRotate(
            shift_limit=0.0625,
            scale_limit=0.1,
            rotate_limit=20,
            p=0.5,
        ))
        transforms_list.insert(4, A.RandomBrightnessContrast(p=0.3))

    return A.Compose(
        transforms_list,
        bbox_params=A.BboxParams(format="pascal_voc", label_fields=["labels"]),
    )


# ============================================================================
# DATASET
# ============================================================================
class FemurDataset(Dataset):
    """
    PyTorch Dataset for femoral fracture detection.

    Loads images and COCO-format annotations with bounding box clipping
    to ensure all annotations remain within image boundaries.
    """

    def __init__(self, img_dir, ann_file, transforms=None):
        self.img_dir    = img_dir
        self.coco       = COCO(ann_file)
        self.ids        = list(sorted(self.coco.imgs.keys()))
        self.transforms = transforms

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        img_id   = self.ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, img_info["file_name"])

        # Load and convert image (BGR → RGB)
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Parse annotations
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns    = self.coco.loadAnns(ann_ids)
        boxes, labels = [], []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            x  = max(0, x)
            y  = max(0, y)
            x2 = min(image.shape[1], x + w)
            y2 = min(image.shape[0], y + h)
            if x2 <= x or y2 <= y:
                continue  # skip degenerate boxes
            boxes.append([x, y, x2, y2])
            labels.append(ann["category_id"])

        if len(boxes) == 0:
            boxes  = np.zeros((0, 4), dtype=np.float32)
            labels = np.zeros((0,),   dtype=np.int64)
        else:
            boxes  = np.array(boxes,  dtype=np.float32)
            labels = np.array(labels, dtype=np.int64)

        if self.transforms:
            try:
                out    = self.transforms(image=image, bboxes=boxes, labels=labels)
                image  = out["image"]
                boxes  = out["bboxes"]
                labels = out["labels"]

                if len(boxes) > 0:
                    boxes  = torch.as_tensor(boxes,  dtype=torch.float32)
                    labels = torch.as_tensor(labels, dtype=torch.int64)
                else:
                    boxes  = torch.zeros((0, 4), dtype=torch.float32)
                    labels = torch.zeros((0,),   dtype=torch.int64)
            except Exception as e:
                print(f"Transform error for {img_path}: {e}")
                boxes  = torch.zeros((0, 4), dtype=torch.float32)
                labels = torch.zeros((0,),   dtype=torch.int64)

        target = {
            "boxes":    boxes,
            "labels":   labels,
            "image_id": torch.tensor([img_id]),
        }
        return image, target


# ============================================================================
# METRICS CALCULATOR
# ============================================================================
class MetricsCalculator:
    """Computes detection and classification metrics."""

    @staticmethod
    def calculate_detection_metrics(model, data_loader, iou_threshold=0.5):
        """
        Compute mAP@0.5 (simplified) and average IoU.

        Returns:
            dict with keys: mAP@0.5, average_iou, num_predictions, num_ground_truths
        """
        model.eval()
        all_predictions, all_ground_truths, ious_list = [], [], []

        with torch.no_grad():
            for images, targets in tqdm(data_loader, desc="Detection Metrics", leave=False):
                images  = [img.to(Config.DEVICE) for img in images]
                outputs = model(images)

                for target, output in zip(targets, outputs):
                    gt_boxes    = target["boxes"].cpu().numpy()
                    gt_labels   = target["labels"].cpu().numpy()
                    pred_boxes  = output["boxes"].cpu().numpy()
                    pred_labels = output["labels"].cpu().numpy()
                    pred_scores = output["scores"].cpu().numpy()

                    # Filter by class-specific confidence threshold
                    if len(pred_labels) > 0:
                        keep = np.array([
                            score >= Config.CONF_THRESH.get(int(lbl), 0.5)
                            for lbl, score in zip(pred_labels, pred_scores)
                        ], dtype=bool)
                        pred_boxes  = pred_boxes[keep]
                        pred_labels = pred_labels[keep]
                        pred_scores = pred_scores[keep]

                    for box, lbl, score in zip(pred_boxes, pred_labels, pred_scores):
                        all_predictions.append({"bbox": box, "label": int(lbl), "score": float(score)})
                    for box, lbl in zip(gt_boxes, gt_labels):
                        all_ground_truths.append({"bbox": box, "label": int(lbl)})

                    # IoU matching
                    if len(gt_boxes) > 0 and len(pred_boxes) > 0:
                        ious     = box_iou(torch.tensor(gt_boxes), torch.tensor(pred_boxes)).numpy()
                        max_ious = ious.max(axis=1)
                        ious_list.extend(max_ious[max_ious >= iou_threshold])

        map_score = (
            len(ious_list) / max(len(all_ground_truths), 1)
            if all_predictions and all_ground_truths else 0.0
        )
        avg_iou = float(np.mean(ious_list)) if ious_list else 0.0

        return {
            "mAP@0.5":         map_score,
            "average_iou":     avg_iou,
            "num_predictions": len(all_predictions),
            "num_ground_truths": len(all_ground_truths),
        }

    @staticmethod
    def calculate_classification_metrics(model, data_loader):
        """
        Treat the highest-confidence detection per image as the classification result.

        Returns:
            tuple: (y_true, y_pred) as numpy arrays
        """
        model.eval()
        y_true, y_pred = [], []

        with torch.no_grad():
            for images, targets in tqdm(data_loader, desc="Classification Metrics", leave=False):
                images  = [img.to(Config.DEVICE) for img in images]
                outputs = model(images)

                for target, output in zip(targets, outputs):
                    gt_labels   = target["labels"].cpu().numpy()
                    pred_labels = output["labels"].cpu().numpy()
                    pred_scores = output["scores"].cpu().numpy()

                    if len(pred_labels) > 0:
                        keep = np.array([
                            score >= Config.CONF_THRESH.get(int(lbl), 0.5)
                            for lbl, score in zip(pred_labels, pred_scores)
                        ], dtype=bool)
                        pred_labels = pred_labels[keep]

                    y_true.append(gt_labels[0]   if len(gt_labels)   > 0 else 0)
                    y_pred.append(pred_labels[0] if len(pred_labels) > 0 else 0)

        return np.array(y_true), np.array(y_pred)


# ============================================================================
# VISUALIZER
# ============================================================================
class Visualizer:
    """Visualization utilities for training metrics and results."""

    @staticmethod
    def plot_training_history(history, save_path=None):
        """Plot a 3×2 grid of training metric curves."""
        fig, axes = plt.subplots(3, 2, figsize=(15, 12))
        fig.suptitle("Training History", fontsize=16, fontweight="bold")

        plots = [
            (axes[0, 0], "loss",          "Training Loss",          "Loss",     "b-"),
            (axes[0, 1], "accuracy",      "Accuracy",               "Accuracy", "g-"),
            (axes[1, 0], "f1",            "F1 Score",               "F1",       "r-"),
            (axes[1, 1], "map",           "mAP@0.5",                "mAP",      "purple"),
            (axes[2, 1], "learning_rate", "Learning Rate",          "LR",       "brown"),
        ]
        for ax, key, title, ylabel, style in plots:
            ax.plot(history[key], style, linewidth=2)
            ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
            ax.grid(True, alpha=0.3)
        if key == "learning_rate":
            axes[2, 1].set_yscale("log")

        axes[2, 0].plot(history["sensitivity"], "orange", linewidth=2, label="Sensitivity")
        axes[2, 0].plot(history["specificity"], "cyan",   linewidth=2, label="Specificity")
        axes[2, 0].set_title("Sensitivity & Specificity")
        axes[2, 0].set_xlabel("Epoch"); axes[2, 0].set_ylabel("Score")
        axes[2, 0].legend(); axes[2, 0].grid(True, alpha=0.3)

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
            print(f"Training history saved to: {save_path}")
        plt.show()

    @staticmethod
    def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix", save_path=None):
        """Plot a 4×4 confusion matrix with AO class labels."""
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
        plt.figure(figsize=(10, 8))
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Oranges",
            xticklabels=list(Config.AO_LABELS.values()),
            yticklabels=list(Config.AO_LABELS.values()),
            cbar_kws={"label": "Count"},
        )
        plt.title(title, fontsize=14, fontweight="bold")
        plt.ylabel("True Label", fontsize=12)
        plt.xlabel("Predicted Label", fontsize=12)
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches="tight")
            print(f"Confusion matrix saved to: {save_path}")
        plt.show()


# ============================================================================
# TRAINING FUNCTION
# ============================================================================
def train_model():
    """
    Run the full training pipeline.

    Returns:
        model: Best-checkpoint Faster R-CNN model.
    """
    print("\n" + "=" * 70)
    print("FEMORAL FRACTURE DETECTION — TRAINING PIPELINE")
    print("=" * 70 + "\n")

    writer = SummaryWriter(Config.TENSORBOARD_LOG_DIR)
    print(f"TensorBoard logs: {Config.TENSORBOARD_LOG_DIR}\n")

    # Datasets
    print("Loading datasets...")
    train_ds = FemurDataset(Config.TRAIN_IMG_DIR, Config.TRAIN_JSON, get_transforms(True))
    val_ds   = FemurDataset(Config.VAL_IMG_DIR,   Config.VAL_JSON,   get_transforms(False))
    print(f"  Training samples:   {len(train_ds)}")
    print(f"  Validation samples: {len(val_ds)}\n")

    collate_fn   = lambda batch: tuple(zip(*batch))
    pin_mem      = Config.DEVICE.type == "cuda"
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True,
                              num_workers=Config.NUM_WORKERS, collate_fn=collate_fn, pin_memory=pin_mem)
    val_loader   = DataLoader(val_ds,   batch_size=Config.BATCH_SIZE, shuffle=False,
                              num_workers=Config.NUM_WORKERS, collate_fn=collate_fn, pin_memory=pin_mem)

    # Model
    print("Initializing model...")
    model      = fasterrcnn_resnet50_fpn_v2(weights="DEFAULT")
    in_feats   = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feats, Config.NUM_CLASSES)
    model.to(Config.DEVICE)
    print(f"Model loaded on: {Config.DEVICE}\n")

    params       = [p for p in model.parameters() if p.requires_grad]
    optimizer    = torch.optim.SGD(params, lr=Config.LEARNING_RATE,
                                   momentum=Config.MOMENTUM, weight_decay=Config.WEIGHT_DECAY)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS)

    history = defaultdict(list)
    best_f1 = 0.0
    start   = datetime.now()

    print("=" * 70)
    print("TRAINING START")
    print("=" * 70 + "\n")

    for epoch in range(Config.EPOCHS):
        # ── Training phase ──────────────────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}", colour="orange")

        for batch_idx, (images, targets) in enumerate(pbar):
            images  = [img.to(Config.DEVICE) for img in images]
            targets = [{k: v.to(Config.DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses    = sum(loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            epoch_loss += losses.item()
            pbar.set_postfix({"loss": f"{losses.item():.4f}"})

            global_step = epoch * len(train_loader) + batch_idx
            writer.add_scalar("Training/Batch_Loss", losses.item(), global_step)

        lr_scheduler.step()
        current_lr = optimizer.param_groups[0]["lr"]
        avg_loss   = epoch_loss / len(train_loader)

        # ── Validation phase ─────────────────────────────────────────────────
        print(f"\nEvaluating epoch {epoch+1}...")
        cm_true, cm_pred = MetricsCalculator.calculate_classification_metrics(model, val_loader)

        accuracy                     = accuracy_score(cm_true, cm_pred)
        precision, recall, f1, _     = precision_recall_fscore_support(
            cm_true, cm_pred, average="macro", zero_division=0
        )

        # Macro-averaged specificity across all 4 classes
        cm_matrix     = confusion_matrix(cm_true, cm_pred, labels=[0, 1, 2, 3])
        specificities = []
        for i in range(len(cm_matrix)):
            tn = cm_matrix.sum() - (cm_matrix[i, :].sum() + cm_matrix[:, i].sum() - cm_matrix[i, i])
            fp = cm_matrix[:, i].sum() - cm_matrix[i, i]
            specificities.append(tn / (tn + fp + 1e-6))
        specificity = float(np.mean(specificities))

        det_metrics = MetricsCalculator.calculate_detection_metrics(model, val_loader)
        map_score   = det_metrics["mAP@0.5"]
        avg_iou     = det_metrics["average_iou"]

        # Store history
        for key, val in [("loss", avg_loss), ("accuracy", accuracy), ("precision", precision),
                         ("sensitivity", recall), ("specificity", specificity), ("f1", f1),
                         ("map", map_score), ("avg_iou", avg_iou), ("learning_rate", current_lr)]:
            history[key].append(val)

        # TensorBoard
        writer.add_scalar("Training/Loss",             avg_loss,    epoch)
        writer.add_scalar("Training/Learning_Rate",    current_lr,  epoch)
        writer.add_scalar("Classification/Accuracy",   accuracy,    epoch)
        writer.add_scalar("Classification/Precision",  precision,   epoch)
        writer.add_scalar("Classification/Sensitivity",recall,      epoch)
        writer.add_scalar("Classification/Specificity",specificity, epoch)
        writer.add_scalar("Classification/F1_Score",   f1,          epoch)
        writer.add_scalar("Detection/mAP@0.5",         map_score,   epoch)
        writer.add_scalar("Detection/Average_IoU",     avg_iou,     epoch)

        # Epoch summary
        print(f"\n{'='*70}\nEpoch {epoch+1}/{Config.EPOCHS} Summary\n{'='*70}")
        print(f"Loss: {avg_loss:.4f} | LR: {current_lr:.6f}")
        print(f"\nClassification Metrics:")
        print(f"  Accuracy:    {accuracy:.4f}")
        print(f"  Precision:   {precision:.4f}")
        print(f"  Sensitivity: {recall:.4f}")
        print(f"  Specificity: {specificity:.4f}")
        print(f"  F1 Score:    {f1:.4f}")
        print(f"\nDetection Metrics:")
        print(f"  mAP@0.5:     {map_score:.4f}")
        print(f"  Avg IoU:     {avg_iou:.4f}")

        # Save best model
        if f1 > best_f1:
            best_f1 = f1
            torch.save({
                "epoch":               epoch,
                "model_state_dict":    model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "f1_score":            f1,
                "metrics": {
                    "accuracy":    accuracy,
                    "precision":   precision,
                    "sensitivity": recall,
                    "specificity": specificity,
                    "map":         map_score,
                    "avg_iou":     avg_iou,
                },
                "config": {
                    "num_classes":   Config.NUM_CLASSES,
                    "image_size":    Config.IMAGE_SIZE,
                    "batch_size":    Config.BATCH_SIZE,
                    "learning_rate": Config.LEARNING_RATE,
                },
            }, Config.MODEL_SAVE_PATH)
            print(f"\n✓ New best model saved — F1: {best_f1:.4f}")

        print(f"{'='*70}\n")

    # Training complete
    elapsed = datetime.now() - start
    print(f"\n{'='*70}\nTRAINING COMPLETE\n{'='*70}")
    print(f"Total time: {elapsed} | Best F1: {best_f1:.4f}\n{'='*70}\n")
    writer.close()

    # Visualizations
    print("Generating visualizations...")
    Visualizer.plot_training_history(history, save_path=os.path.join(Config.VIZ_DIR, "training_history.png"))

    # Final evaluation with best checkpoint
    print("\nFinal evaluation with best model...")
    ckpt = torch.load(Config.MODEL_SAVE_PATH, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])

    final_true, final_pred = MetricsCalculator.calculate_classification_metrics(model, val_loader)
    Visualizer.plot_confusion_matrix(
        final_true, final_pred,
        title="Validation Confusion Matrix",
        save_path=os.path.join(Config.VIZ_DIR, "confusion_matrix.png"),
    )

    # Print final summary
    print(f"\n{'='*70}\nFINAL VALIDATION METRICS (Best Checkpoint — Epoch {ckpt['epoch']+1})\n{'='*70}")
    m = ckpt["metrics"]
    print(f"  Accuracy:    {m['accuracy']:.4f}")
    print(f"  Precision:   {m['precision']:.4f}")
    print(f"  Sensitivity: {m['sensitivity']:.4f}")
    print(f"  Specificity: {m['specificity']:.4f}")
    print(f"  F1 Score:    {ckpt['f1_score']:.4f}")
    print(f"  mAP@0.5:     {m['map']:.4f}")
    print(f"  Avg IoU:     {m['avg_iou']:.4f}")
    print("=" * 70)

    # Save text summary
    summary_path = os.path.join(Config.RUN_DIR, "training_summary.txt")
    with open(summary_path, "w") as fout:
        fout.write(f"Training Time: {elapsed}\n")
        fout.write(f"Best Epoch:    {ckpt['epoch']+1}\n")
        fout.write(f"F1 Score:      {ckpt['f1_score']:.4f}\n\n")
        for k, v in m.items():
            fout.write(f"{k}: {v:.4f}\n")
    print(f"\nTraining summary saved to: {summary_path}")

    return model


# ============================================================================
# ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    trained_model = train_model()


## 5️⃣ TensorBoard Monitoring

### Purpose
Visualize training progress in real-time using the TensorBoard dashboard.

### Logged Scalars

| Tag | Description |
|-----|-------------|
| `Training/Loss` | Average training loss per epoch |
| `Training/Learning_Rate` | Current LR (CosineAnnealingLR scheduler) |
| `Training/Batch_Loss` | Per-batch training loss |
| `Classification/Accuracy` | Validation accuracy per epoch |
| `Classification/Precision` | Validation precision (macro) per epoch |
| `Classification/Sensitivity` | Validation recall / sensitivity (macro) per epoch |
| `Classification/Specificity` | Validation specificity (macro) per epoch |
| `Classification/F1_Score` | Validation F1-score (macro) per epoch |
| `Detection/mAP@0.5` | Mean Average Precision at IoU ≥ 0.5 per epoch |
| `Detection/Average_IoU` | Average IoU of matched detections per epoch |

### How to Use
1. Run the cell below after training has started
2. Navigate the **SCALARS** tab to track metrics over epochs
3. Refresh the page to see updated metrics during training

---


In [ ]:
# Load the TensorBoard extension
%load_ext tensorboard

print("TensorBoard extension loaded")


In [ ]:
# Launch TensorBoard — update the log directory path if needed
%tensorboard --logdir "/content/drive/MyDrive/Thesis/Thesis 2/System Files/system/logs"


In [ ]:
# Export TensorBoard scalar plots as PNG files
import os
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing import event_accumulator

LOG_DIR    = "/content/drive/MyDrive/Thesis/Thesis 2/System Files/system/logs"
OUTPUT_DIR = os.path.join(LOG_DIR, "visualizations/tensorboard_metrics_png")
os.makedirs(OUTPUT_DIR, exist_ok=True)

ea = event_accumulator.EventAccumulator(LOG_DIR)
ea.Reload()

for tag in ea.Tags()["scalars"]:
    events = ea.Scalars(tag)
    steps  = [e.step  for e in events]
    values = [e.value for e in events]

    parts   = tag.split("/")
    section = parts[0].capitalize() if len(parts) > 1 else ""
    metric  = parts[-1].replace("_", " ").title()
    title   = f"{section}/{metric}" if section else metric

    fig, ax = plt.subplots(figsize=(9, 6))
    is_dense = len(steps) > 100
    ax.plot(steps, values, color="#E87722",
            linewidth=1.2 if is_dense else 2.0,
            marker="o", markersize=2.5 if is_dense else 4, markeredgewidth=0)

    ax.set_title(title, fontsize=16, fontweight="bold", pad=12)
    ax.set_xlabel("Epoch", fontsize=13)
    ax.set_ylabel(parts[-1], fontsize=13)
    ax.set_axisbelow(True)
    ax.grid(True, linestyle="--", linewidth=0.8, color="#cccccc")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")
    plt.tight_layout()

    filename  = tag.replace("/", "_") + ".png"
    save_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved: {filename}")

print("\nAll TensorBoard graphs exported.")


In [ ]:
# Display exported TensorBoard plots grouped by metric category
import os
import matplotlib.pyplot as plt
from PIL import Image

IMAGE_DIR = "/content/drive/MyDrive/Thesis/Thesis 2/System Files/system/logs/visualizations/tensorboard_metrics_png"


def show_group(title, file_list, ncols=2, figsize=(16, 8)):
    if not file_list:
        return
    nrows = (len(file_list) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten()
    fig.suptitle(title, fontsize=18, fontweight="bold", y=1.02)

    for ax, fname in zip(axes, file_list):
        img = Image.open(os.path.join(IMAGE_DIR, fname))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(fname.replace("_", " ").replace(".png", ""), fontsize=11)

    for ax in axes[len(file_list):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


png_files = sorted(f for f in os.listdir(IMAGE_DIR) if f.endswith(".png"))

show_group("Training Metrics (Loss and Learning Rate)",
           [f for f in png_files if f.startswith("Training")],
           ncols=2, figsize=(16, 10))

show_group("Classification Performance Metrics",
           [f for f in png_files if f.startswith("Classification")],
           ncols=3, figsize=(18, 10))

show_group("Detection Performance Metrics",
           [f for f in png_files if f.startswith("Detection")],
           ncols=2, figsize=(16, 6))


## 6️⃣ Test Set Evaluation

### Purpose
Evaluate the trained model on the **held-out test set** to obtain final, unbiased performance metrics.

### Why the Test Set?
- The validation set was used during training for model selection (best F1-score)
- The test set is **completely unseen** during training, providing an honest estimate of real-world performance

### Evaluation Process

**Step 1 — Model Loading**
Loads the best checkpoint (`best_model.pth`) and reinitializes the Faster R-CNN architecture with 4-class output.

**Step 2 — Inference**
Processes all test images with the same preprocessing pipeline used during training (no augmentation), then applies class-specific confidence thresholds to filter predictions.

**Step 3 — Metrics**

| Metric | Formula | Target |
|--------|---------|--------|
| Accuracy | (TP + TN) / Total | ≥ 0.85 |
| Precision | TP / (TP + FP), macro | ≥ 0.85 |
| Sensitivity | TP / (TP + FN), macro | ≥ 0.85 |
| Specificity | TN / (TN + FP), macro | ≥ 0.85 |
| F1-Score | harmonic mean of precision & sensitivity | ≥ 0.85 |
| mAP@0.5 (simplified) | matched detections (IoU ≥ 0.5) / ground truths | ≥ 0.80 |
| Average IoU | mean IoU of matched detections | — |

**Step 4 — Confusion Matrix**
A 4×4 matrix (rows = true labels, columns = predicted labels) is saved and displayed.

---


In [ ]:
# Test Set Evaluation
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

print("=" * 70)
print("TEST SET EVALUATION")
print("=" * 70)

if not os.path.exists(Config.TEST_IMG_DIR) or not os.path.exists(Config.TEST_JSON):
    print(f"Warning: Test set not found at {Config.TEST_IMG_DIR}")
    print("Please verify your directory structure.")
else:
    print(f"Loading test set from: {Config.TEST_IMG_DIR}")

    test_ds = FemurDataset(Config.TEST_IMG_DIR, Config.TEST_JSON, get_transforms(False))
    test_loader = DataLoader(
        test_ds,
        batch_size=Config.BATCH_SIZE,
        shuffle=False,
        num_workers=Config.NUM_WORKERS,
        collate_fn=lambda b: tuple(zip(*b)),
    )
    print(f"Test samples: {len(test_ds)}")

    # Load best model
    print("\nLoading best model...")
    if os.path.exists(Config.MODEL_SAVE_PATH):
        checkpoint = torch.load(Config.MODEL_SAVE_PATH, weights_only=False)
        model = fasterrcnn_resnet50_fpn_v2(weights=None)
        in_feats = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_feats, Config.NUM_CLASSES)
        model.load_state_dict(checkpoint["model_state_dict"])
        model.to(Config.DEVICE)
        model.eval()
        print("Model loaded successfully")
    else:
        print("Error: No saved model found. Please run training first.")
        model = None

    if model is not None:
        print("\nRunning test set evaluation...")

        y_true, y_pred = MetricsCalculator.calculate_classification_metrics(model, test_loader)

        accuracy            = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="macro", zero_division=0
        )

        # Macro-averaged specificity
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
        specificities = []
        for i in range(len(cm)):
            tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
            fp = cm[:, i].sum() - cm[i, i]
            specificities.append(tn / (tn + fp + 1e-6))
        specificity = float(np.mean(specificities))

        det_metrics = MetricsCalculator.calculate_detection_metrics(model, test_loader)
        map_score   = det_metrics["mAP@0.5"]
        avg_iou     = det_metrics["average_iou"]

        # ── Print results ────────────────────────────────────────────────────
        print("\n" + "=" * 70)
        print("OFFICIAL TEST SET RESULTS")
        print("=" * 70)
        print("\nClassification Metrics:")
        print(f"  Accuracy:    {accuracy:.4f}")
        print(f"  Precision:   {precision:.4f}")
        print(f"  Sensitivity: {recall:.4f}")
        print(f"  Specificity: {specificity:.4f}")
        print(f"  F1 Score:    {f1:.4f}")
        print("\nDetection Metrics:")
        print(f"  mAP@0.5:     {map_score:.4f}")
        print(f"  Avg IoU:     {avg_iou:.4f}")
        print("=" * 70)

        # Per-class breakdown
        class_names = list(Config.AO_LABELS.values())
        pc_p, pc_r, pc_f1, _ = precision_recall_fscore_support(
            y_true, y_pred, labels=[0, 1, 2, 3], zero_division=0
        )
        print("\nPer-Class Metrics:")
        print(f"{'Class':<20} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
        print("-" * 56)
        for i, name in enumerate(class_names):
            print(f"{name:<20} {pc_p[i]:<12.4f} {pc_r[i]:<12.4f} {pc_f1[i]:<12.4f}")

        # Confusion matrix
        Visualizer.plot_confusion_matrix(
            y_true, y_pred,
            title="Test Set Confusion Matrix",
            save_path=os.path.join(Config.VIZ_DIR, "test_confusion_matrix.png"),
        )

        # Save results
        results_path = os.path.join(Config.RUN_DIR, "test_results.txt")
        with open(results_path, "w") as fout:
            fout.write("TEST SET RESULTS\n" + "=" * 70 + "\n\n")
            fout.write(f"Accuracy:    {accuracy:.4f}\n")
            fout.write(f"Precision:   {precision:.4f}\n")
            fout.write(f"Sensitivity: {recall:.4f}\n")
            fout.write(f"Specificity: {specificity:.4f}\n")
            fout.write(f"F1 Score:    {f1:.4f}\n")
            fout.write(f"mAP@0.5:     {map_score:.4f}\n")
            fout.write(f"Avg IoU:     {avg_iou:.4f}\n")
        print(f"\nTest results saved to: {results_path}")


## 7️⃣ Interactive Gradio Interface (Optional)

### Purpose
Launch a **web-based interactive interface** for real-time fracture detection and classification.

### Features
- Drag-and-drop image upload
- Optional CLAHE contrast enhancement preview
- Real-time Faster R-CNN inference with annotated bounding boxes
- Per-class confidence threshold sliders
- Shareable public link (valid for 72 hours)

### Confidence Thresholds (Deployment)

| Class | Threshold | Note |
|-------|-----------|------|
| 32-A Simple | 0.55 | Slightly lower than training for higher sensitivity |
| 32-B Wedge | 0.45 | Same as training |
| 32-C Complex | 0.55 | Slightly lower than training for higher sensitivity |

### How to Use
1. Run the cell below — model loading takes ~10–20 seconds
2. Copy the public Gradio link that appears
3. Upload a veterinary radiograph (JPEG or PNG)
4. Optionally click **Apply CLAHE** to preview enhanced contrast
5. Click **Detect Fractures** to run inference
6. Review annotated output and detection summary

---


In [ ]:
# Gradio Demo — Femoral Shaft Fracture Detection
!pip install -q gradio albumentations==1.3.1

import os
import cv2
import torch
import numpy as np
import albumentations as A
import gradio as gr
import torchvision
import warnings

from albumentations.pytorch import ToTensorV2
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

warnings.filterwarnings("ignore", category=DeprecationWarning)


# ─── Config ──────────────────────────────────────────────────────────────────
class GradioConfig:
    MODEL_PATH  = "/content/drive/MyDrive/Thesis/Thesis 2-3/System Files/system/models/model 7 (best model)/best_model.pth"
    DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_CLASSES = 4
    IMAGE_SIZE  = 1024

    CONF_THRESH = {
        1: 0.55,  # 32-A Simple
        2: 0.45,  # 32-B Wedge
        3: 0.55,  # 32-C Complex
    }
    AO_LABELS = {
        1: "32-A (Simple)",
        2: "32-B (Wedge)",
        3: "32-C (Complex)",
    }
    BOX_COLORS = {
        1: (255, 193,   7),   # amber  — 32-A
        2: (255, 159,  67),   # orange — 32-B
        3: (220,  53,  69),   # red    — 32-C
    }


# ─── Model Loading ───────────────────────────────────────────────────────────
def load_model():
    model = fasterrcnn_resnet50_fpn_v2(weights=None)
    in_feats = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feats, GradioConfig.NUM_CLASSES)

    ckpt = torch.load(GradioConfig.MODEL_PATH, map_location=GradioConfig.DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt)
    model.to(GradioConfig.DEVICE)
    model.eval()
    return model


model = load_model()
print("Model loaded successfully.")


# ─── Transforms ──────────────────────────────────────────────────────────────
def get_inference_transform():
    return A.Compose([
        A.LongestMaxSize(max_size=GradioConfig.IMAGE_SIZE),
        A.PadIfNeeded(min_height=GradioConfig.IMAGE_SIZE, min_width=GradioConfig.IMAGE_SIZE,
                      border_mode=cv2.BORDER_CONSTANT, value=0),
        A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=1.0),
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2(),
    ])


# ─── CLAHE Preview ───────────────────────────────────────────────────────────
def apply_clahe_preview(image):
    if image is None:
        return None, "Please upload an X-ray image first."

    img = np.array(image)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB), "CLAHE enhancement applied successfully."


# ─── Inference ───────────────────────────────────────────────────────────────
def run_inference(image, conf_32a, conf_32b, conf_32c):
    if image is None:
        return None, "Please upload an X-ray image to begin detection."

    GradioConfig.CONF_THRESH[1] = conf_32a
    GradioConfig.CONF_THRESH[2] = conf_32b
    GradioConfig.CONF_THRESH[3] = conf_32c

    original = np.array(image)
    orig_h, orig_w = original.shape[:2]

    tensor_img = get_inference_transform()(image=original)["image"].unsqueeze(0).to(GradioConfig.DEVICE)

    with torch.no_grad():
        output = model(tensor_img)[0]

    keep   = torchvision.ops.nms(output["boxes"], output["scores"], iou_threshold=0.3)
    boxes  = output["boxes"][keep].cpu().numpy()
    labels = output["labels"][keep].cpu().numpy()
    scores = output["scores"][keep].cpu().numpy()

    scale  = GradioConfig.IMAGE_SIZE / max(orig_h, orig_w)
    pad_y  = (GradioConfig.IMAGE_SIZE - int(orig_h * scale)) // 2
    pad_x  = (GradioConfig.IMAGE_SIZE - int(orig_w * scale)) // 2

    result_img = original.copy()
    detections = []

    for box, label, score in zip(boxes, labels, scores):
        label = int(label)
        if label == 0:
            continue
        if score < GradioConfig.CONF_THRESH.get(label, 0.5):
            continue

        x1 = max(0, int((box[0] - pad_x) / scale))
        y1 = max(0, int((box[1] - pad_y) / scale))
        x2 = min(orig_w, int((box[2] - pad_x) / scale))
        y2 = min(orig_h, int((box[3] - pad_y) / scale))

        color     = GradioConfig.BOX_COLORS[label]
        cls_name  = GradioConfig.AO_LABELS[label]
        text      = f"{cls_name} {score:.1%}"
        detections.append(f"{cls_name}: {score:.1%}")

        cv2.rectangle(result_img, (x1, y1), (x2, y2), color, 4)

        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
        cv2.rectangle(result_img, (x1, y1 - th - 15), (x1 + tw + 10, y1), color, -1)
        cv2.putText(result_img, text, (x1 + 5, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    if detections:
        summary  = "DETECTED FRACTURES:\n" + "─" * 40 + "\n"
        summary += "\n".join(f"• {d}" for d in detections)
        summary += "\n" + "─" * 40
        summary += f"\nTotal Detections: {len(detections)}"
    else:
        summary = "No fractures detected above confidence threshold."

    return result_img, summary


# ─── Gradio UI ───────────────────────────────────────────────────────────────
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Google+Sans:wght@400;500;700&family=Roboto:wght@300;400;500&display=swap');

* { box-sizing: border-box; }

.gradio-container {
    font-family: 'Roboto', 'Google Sans', sans-serif !important;
    max-width: 1200px !important;
    margin: 0 auto !important;
    background: #ffffff !important;
    color: #202124 !important;
}

.nav-bar {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 0 24px;
    height: 64px;
    border-bottom: 1px solid #e8eaed;
    background: #fff;
    position: sticky;
    top: 0;
    z-index: 100;
}
.nav-logo { display: flex; align-items: center; gap: 10px; }
.nav-logo-text {
    font-family: 'Google Sans', sans-serif;
    font-size: 18px;
    font-weight: 500;
    color: #202124;
    letter-spacing: -0.2px;
}
.nav-logo-text span { color: #4285f4; }
.nav-badge {
    font-size: 10px;
    font-weight: 500;
    color: #fff;
    background: #34a853;
    padding: 2px 7px;
    border-radius: 10px;
    letter-spacing: 0.3px;
    text-transform: uppercase;
}

.hero {
    text-align: center;
    padding: 56px 24px 40px;
    background: #fff;
    display: flex;
    flex-direction: column;
    align-items: center;
}
.hero-eyebrow {
    display: inline-flex;
    align-items: center;
    gap: 6px;
    font-size: 12px;
    font-weight: 500;
    color: #4285f4;
    background: #e8f0fe;
    padding: 4px 12px;
    border-radius: 16px;
    margin-bottom: 20px;
    letter-spacing: 0.4px;
    text-transform: uppercase;
}
.hero h1 {
    font-family: 'Google Sans', sans-serif;
    font-size: 2.6rem;
    font-weight: 400;
    color: #202124;
    margin: 0 0 16px;
    line-height: 1.25;
    letter-spacing: -0.5px;
    white-space: nowrap;
}
.hero h1 strong {
    font-weight: 700;
    background: linear-gradient(90deg, #4285f4, #34a853);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
}
.hero-sub {
    font-size: 1.05rem;
    color: #5f6368;
    max-width: 640px;
    margin: 0 auto 32px;
    line-height: 1.6;
    font-weight: 400;
    text-align: center;
    white-space: nowrap;
}

.section-divider { height: 1px; background: #e8eaed; margin: 0 24px; }

.info-grid {
    display: grid;
    grid-template-columns: 1fr 1fr 1fr;
    gap: 12px;
    margin: 24px 24px 0;
}
.info-tile {
    background: #f8f9fa;
    border: 1px solid #e8eaed;
    border-radius: 10px;
    padding: 16px 18px;
}
.info-tile-label {
    font-size: 11px;
    font-weight: 500;
    color: #80868b;
    text-transform: uppercase;
    letter-spacing: 0.5px;
    margin-bottom: 6px;
}
.info-tile-value {
    font-family: 'Google Sans', sans-serif;
    font-size: 15px;
    font-weight: 500;
    color: #202124;
    line-height: 1.4;
}
.info-tile-dot {
    display: inline-block;
    width: 8px; height: 8px;
    border-radius: 50%;
    margin-right: 6px;
    vertical-align: middle;
}

.steps-row {
    display: flex;
    margin: 24px 24px 0;
    background: #f8f9fa;
    border: 1px solid #e8eaed;
    border-radius: 10px;
    overflow: hidden;
}
.step-item {
    flex: 1;
    padding: 14px 16px;
    border-right: 1px solid #e8eaed;
    display: flex;
    align-items: flex-start;
    gap: 10px;
}
.step-item:last-child { border-right: none; }
.step-num {
    width: 22px; height: 22px;
    background: #4285f4;
    border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    font-size: 11px; font-weight: 700; color: #fff;
    flex-shrink: 0; margin-top: 1px;
}
.step-text { font-size: 12px; color: #3c4043; line-height: 1.5; }

.disclaimer-bar {
    margin: 16px 24px 0;
    background: #fef9e3;
    border: 1px solid #fdd663;
    border-radius: 8px;
    padding: 10px 16px;
    display: flex;
    align-items: flex-start;
    gap: 10px;
    font-size: 12.5px;
    color: #594300;
    line-height: 1.5;
}

.adv-header {
    margin: 28px 24px 0;
    padding: 0 0 12px;
    border-bottom: 1px solid #e8eaed;
    display: flex;
    align-items: center;
    gap: 8px;
}
.adv-title {
    font-family: 'Google Sans', sans-serif;
    font-size: 14px; font-weight: 500; color: #202124;
}
.adv-sub { font-size: 12px; color: #80868b; margin-left: auto; }

.footer-bar {
    margin: 32px 0 0;
    padding: 24px;
    border-top: 1px solid #e8eaed;
    display: flex;
    align-items: center;
    justify-content: space-between;
    flex-wrap: wrap;
    gap: 12px;
}
.footer-left { font-size: 12px; color: #80868b; line-height: 1.7; }
.footer-left strong { color: #3c4043; font-weight: 500; }
.footer-right { display: flex; gap: 6px; flex-wrap: wrap; }
.tech-pill {
    font-size: 11px; font-weight: 500;
    color: #3c4043; background: #f1f3f4;
    border: 1px solid #e8eaed;
    padding: 3px 10px; border-radius: 12px;
}

.gradio-container .gr-button { border-radius: 6px !important; font-size: 14px !important; font-weight: 500 !important; }
.gradio-container label { font-size: 13px !important; color: #3c4043 !important; font-weight: 500 !important; }
.gradio-container input, .gradio-container textarea {
    border-color: #e8eaed !important; border-radius: 6px !important;
    font-size: 13px !important; background: #fff !important;
}
.gradio-container input:focus, .gradio-container textarea:focus {
    border-color: #4285f4 !important;
    box-shadow: 0 0 0 2px rgba(66,133,244,0.15) !important;
    outline: none !important;
}
.upload-zone {
    border: 2px dashed #dadce0 !important;
    border-radius: 10px !important;
    background: #fafafa !important;
    transition: border-color 0.2s !important;
}
.upload-zone:hover { border-color: #4285f4 !important; }
"""

nav_html = """
<div class="nav-bar">
    <div class="nav-logo">
        <div class="nav-logo-text">VetFracture<span>AI</span></div>
        <span class="nav-badge">Research</span>
    </div>
</div>
"""

hero_html = """
<div class="hero">
    <div class="hero-eyebrow">Veterinary Radiology · Deep Learning</div>
    <h1>Femoral Shaft <strong>Fracture Detection</strong></h1>
    <p class="hero-sub">Upload a veterinary X-ray and get instant AI-powered AO/OTA fracture classification using Faster R-CNN.</p>
</div>
<div class="section-divider"></div>
"""

info_html = """
<div class="info-grid">
    <div class="info-tile">
        <div class="info-tile-label">Model</div>
        <div class="info-tile-value">Faster R-CNN<br><span style="font-size:12px;color:#80868b;font-weight:400">ResNet50-FPN v2</span></div>
    </div>
    <div class="info-tile">
        <div class="info-tile-label">Classification</div>
        <div class="info-tile-value">
            <span><span class="info-tile-dot" style="background:#f59e0b"></span>32-A Simple</span><br>
            <span><span class="info-tile-dot" style="background:#f97316"></span>32-B Wedge</span><br>
            <span><span class="info-tile-dot" style="background:#ef4444"></span>32-C Complex</span>
        </div>
    </div>
    <div class="info-tile">
        <div class="info-tile-label">Input</div>
        <div class="info-tile-value">X-ray image<br><span style="font-size:12px;color:#80868b;font-weight:400">JPG · PNG · JPEG</span></div>
    </div>
</div>

<div class="steps-row">
    <div class="step-item"><div class="step-num">1</div><div class="step-text"><strong>Upload</strong> your X-ray image</div></div>
    <div class="step-item"><div class="step-num">2</div><div class="step-text"><strong>Enhance</strong> with CLAHE (optional)</div></div>
    <div class="step-item"><div class="step-num">3</div><div class="step-text"><strong>Detect</strong> fractures with AI</div></div>
    <div class="step-item"><div class="step-num">4</div><div class="step-text"><strong>Review</strong> annotated results</div></div>
</div>

<div class="disclaimer-bar">
    <span style="display:inline-flex;align-items:center;flex-shrink:0;margin-top:1px;">
        <svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="#92400e" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
            <path d="M10.29 3.86L1.82 18a2 2 0 0 0 1.71 3h16.94a2 2 0 0 0 1.71-3L13.71 3.86a2 2 0 0 0-3.42 0z"/>
            <line x1="12" y1="9" x2="12" y2="13"/><line x1="12" y1="17" x2="12.01" y2="17"/>
        </svg>
    </span>
    <span><strong>Medical Disclaimer:</strong> This system is for research and educational purposes only and should not replace professional veterinary or medical diagnosis. Always consult a qualified clinician for clinical decisions.</span>
</div>
"""

adv_html = """
<div class="adv-header">
    <span style="display:inline-flex;align-items:center;">
        <svg width="15" height="15" viewBox="0 0 24 24" fill="none" stroke="#5f6368" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
            <circle cx="12" cy="12" r="3"/>
            <path d="M19.4 15a1.65 1.65 0 0 0 .33 1.82l.06.06a2 2 0 0 1-2.83 2.83l-.06-.06a1.65 1.65 0 0 0-1.82-.33 1.65 1.65 0 0 0-1 1.51V21a2 2 0 0 1-4 0v-.09A1.65 1.65 0 0 0 9 19.4a1.65 1.65 0 0 0-1.82.33l-.06.06a2 2 0 0 1-2.83-2.83l.06-.06A1.65 1.65 0 0 0 4.68 15a1.65 1.65 0 0 0-1.51-1H3a2 2 0 0 1 0-4h.09A1.65 1.65 0 0 0 4.6 9a1.65 1.65 0 0 0-.33-1.82l-.06-.06a2 2 0 0 1 2.83-2.83l.06.06A1.65 1.65 0 0 0 9 4.68a1.65 1.65 0 0 0 1-1.51V3a2 2 0 0 1 4 0v.09a1.65 1.65 0 0 0 1 1.51 1.65 1.65 0 0 0 1.82-.33l.06-.06a2 2 0 0 1 2.83 2.83l-.06.06A1.65 1.65 0 0 0 19.4 9a1.65 1.65 0 0 0 1.51 1H21a2 2 0 0 1 0 4h-.09a1.65 1.65 0 0 0-1.51 1z"/>
        </svg>
    </span>
    <span class="adv-title">Advanced Settings</span>
    <span class="adv-sub">Adjust confidence thresholds per fracture type</span>
</div>
"""

footer_html = """
<div class="footer-bar">
    <div class="footer-left">
        <strong>Developed by:</strong> Alvendher Joy I. Francisco <br>
        Mapúa University · Academic Year 2025–2026 · © 2026 Research Prototype
    </div>
    <div class="footer-right">
        <span class="tech-pill">PyTorch</span>
        <span class="tech-pill">Faster R-CNN</span>
        <span class="tech-pill">Gradio</span>
        <span class="tech-pill">OpenCV</span>
        <span class="tech-pill">Albumentations</span>
    </div>
</div>
"""

with gr.Blocks(
    title="VetFractureAI — Femoral Shaft Fracture Detection",
    css=custom_css,
    theme=gr.themes.Base(
        primary_hue="blue",
        neutral_hue="gray",
        font=gr.themes.GoogleFont("Roboto")
    )
) as demo:

    gr.HTML(nav_html)
    gr.HTML(hero_html)
    gr.HTML(info_html)

    gr.HTML("""
    <div style="margin:28px 24px 0;">
        <div class="card-header" style="background:#fff;border:1px solid #e8eaed;border-radius:12px 12px 0 0;border-bottom:1px solid #f1f3f4;padding:20px 24px 16px;display:flex;align-items:center;gap:10px;">
            <div style="width:32px;height:32px;background:#e8f0fe;border-radius:8px;display:flex;align-items:center;justify-content:center;">
                <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="#1a73e8" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
                    <rect x="3" y="3" width="18" height="18" rx="2"/>
                    <line x1="3" y1="9" x2="21" y2="9"/><line x1="3" y1="15" x2="21" y2="15"/>
                    <line x1="9" y1="3" x2="9" y2="21"/><line x1="15" y1="3" x2="15" y2="21"/>
                </svg>
            </div>
            <div>
                <div style="font-family:'Google Sans',sans-serif;font-size:15px;font-weight:500;color:#202124;">Detection Workspace</div>
                <div style="font-size:12px;color:#80868b;margin-top:1px;">Upload an X-ray image and run fracture analysis</div>
            </div>
        </div>
    </div>
    """)

    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            input_image = gr.Image(
                type="pil",
                label="Upload X-ray Image",
                height=420,
                elem_classes="upload-zone"
            )
            with gr.Row():
                clahe_btn  = gr.Button("Apply CLAHE",      variant="secondary", size="lg", scale=1)
                detect_btn = gr.Button("Detect Fractures", variant="primary",   size="lg", scale=2)
                clear_btn  = gr.Button("Clear",            variant="secondary", size="lg", scale=1)
            clahe_status = gr.Textbox(
                label="",
                placeholder="Status messages will appear here...",
                lines=1,
                interactive=False,
                show_label=False
            )

        with gr.Column(scale=1):
            output_image = gr.Image(
                type="numpy",
                label="Detection Results",
                height=420
            )
            output_text = gr.Textbox(
                label="Detection Summary",
                lines=7,
                max_lines=12,
                placeholder="Detection results will appear here after analysis..."
            )

    gr.HTML(adv_html)

    with gr.Row():
        conf_32a = gr.Slider(0.0, 1.0, value=0.55, step=0.05,
                             label="32-A (Simple) threshold",
                             info="Confidence required to show a 32-A detection")
        conf_32b = gr.Slider(0.0, 1.0, value=0.45, step=0.05,
                             label="32-B (Wedge) threshold",
                             info="Confidence required to show a 32-B detection")
        conf_32c = gr.Slider(0.0, 1.0, value=0.55, step=0.05,
                             label="32-C (Complex) threshold",
                             info="Confidence required to show a 32-C detection")

    gr.HTML(footer_html)

    clahe_btn.click(fn=apply_clahe_preview, inputs=input_image,
                    outputs=[input_image, clahe_status])
    detect_btn.click(fn=run_inference,
                     inputs=[input_image, conf_32a, conf_32b, conf_32c],
                     outputs=[output_image, output_text])
    clear_btn.click(fn=lambda: (None, None, ""),
                    outputs=[input_image, output_image, clahe_status])


demo.queue()
demo.launch(share=True, debug=False, show_error=True)

## Appendix

### A. Methodology Summary

| Step | Detail |
|------|--------|
| Dataset format | COCO JSON with train / val / test splits |
| Preprocessing | BGR→RGB, LongestMaxSize(1024), PadIfNeeded(1024×1024, black), CLAHE(clip=3.0, grid=8×8, p=1.0), Normalize(0.5, 0.5) |
| Augmentation (train) | HorizontalFlip(0.5), ShiftScaleRotate(±6.25%, ±10%, ±20°, p=0.5), RandomBrightnessContrast(0.3) |
| Model | Faster R-CNN ResNet-50 FPN V2 — 4 classes (background + 3 AO types) |
| Optimizer | SGD — lr=0.007, momentum=0.9, weight_decay=0.004 |
| LR Scheduler | CosineAnnealingLR — T_max=50 |
| Batch size / Epochs | 16 / 50 |
| Best model criterion | Highest validation F1-score (macro) |
| Evaluation | Classification: Accuracy, Precision, Sensitivity, Specificity, F1 (macro) · Detection: mAP@0.5 (simplified), Avg IoU |

### B. Performance Targets

| Metric | Target | Clinical Significance |
|--------|--------|-----------------------|
| Accuracy | ≥ 0.85 | Overall diagnostic reliability |
| Precision | ≥ 0.85 | Minimizes false positives |
| Sensitivity | ≥ 0.85 | Minimizes missed fractures |
| Specificity | ≥ 0.85 | Correctly identifies non-fractured cases |
| F1-Score | ≥ 0.85 | Balanced detection performance |
| mAP@0.5 | ≥ 0.80 | Fracture localization quality |

### C. Troubleshooting

**Out of Memory (OOM)** — Reduce `Config.BATCH_SIZE` to 8, 4, or 2.

**Dataset not found** — Verify all paths in the `Config` class match your Google Drive structure exactly.

**TensorBoard not showing metrics** — Ensure the `--logdir` path matches `Config.TENSORBOARD_LOG_DIR`.

**Low validation accuracy** — Try reducing `LEARNING_RATE` to 0.002–0.001, or increase augmentation.

**CUDA OOM during inference** — Reduce `batch_size` to 1 in the test evaluation DataLoader.

### D. Directory Reference

```
Google Drive/MyDrive/Thesis/Thesis 2/System Files/system/
├── images/
│   ├── train/       ← training radiographs
│   ├── val/         ← validation radiographs
│   └── test/        ← test radiographs
├── annotations/
│   ├── instances_train.json
│   ├── instances_val.json
│   └── instances_test.json
├── models/
│   └── best_model.pth        ← saved by training pipeline
└── logs/
    ├── fracture_detection_*/  ← TensorBoard event files
    ├── visualizations/        ← confusion matrices, training curves
    ├── training_summary.txt
    └── test_results.txt
```
